# validate_paper_metrics — MobileNetV2 Nano (paper replica: FP32 and Brevitas QAT)

Derived from `code/train/mobilenet/validate_metrics.ipynb`. The DFire-mini, Clouds and dehazing parts were removed; only the paper test set and the two paper models are kept.

**Select the model** with the environment variable `REPLICA_MODEL` (read by `config.py`), and run it in the conda environment of the original training run:

| `REPLICA_MODEL` | Paper result | Historical run | Checkpoint (epoch) | Conda env |
|---|---|---|---|---|
| `fp32` (default) | Table 5 "MobileNetV2 Nano", Table 7 FP32, Table 8 FP32 | `experiments/test_v04_mini_resnet_70k_full_ds` | `MY_MBLNET_V2_classifier__best_mean_F1.pt` (86) | `pytorch_23` |
| `brevitas` | Table 5 "Quantized - MobileNetV2 Nano", Table 7 Quantized, Table 8 "Quantized software" | `experiments_brevitas/test_v05_mini_resnet_70k_full_ds` | `MY_MBLNET_V2_RESNET_classifier__best_mean_F1.pt` (91) | `pytorch_brevitas` |

Weights are local byte-identical copies in `weights/` (sha256 checked below). Working directory: this folder.

Two evaluations:
- **A. Training-identical protocol.** `modules.val_epoch.eval_fn` with `get_val_loader()` (batch 64, `drop_last=True`, i.e. 24,320 of the 24,371 test images), exactly as in the training notebooks. It must reproduce the logged values of the selected epoch.
- **B. Independent per-image evaluation.** Full test set (`drop_last=False`, 24,371 images); TP/FP/TN/FN computed with numpy. Restricted to the first 24,320 images, it must match A.

In [ ]:
# Safety: block any write to ~/uav or code/ (code_review/common/write_guard.py)
import sys, os
from pathlib import Path
# code_review/ = first parent folder that contains common/ (this notebook runs from its own folder)
sys.path.insert(0, str(next(p for p in Path.cwd().resolve().parents if (p / 'common' / '__init__.py').exists())))
from common import write_guard
write_guard.install()
sys.dont_write_bytecode = True

In [ ]:
import json, hashlib, datetime, subprocess, importlib.metadata as md
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

import config
from modules.dataloaders import get_val_loader
from modules.utils import load_checkpoint
from modules.loss import BCE_LOSS
import modules.val_epoch as val_epoch

if config.REPLICA_MODEL == 'fp32':
    from modules.model_mobilenetv2_mini_Resnet import MobileNetV2_MINI_RESNET
else:
    from modules.brevitas.model_mobilenetv2_mini_Resnet_Brevitas import MobileNetV2_MINI_RESNET

DEVICE = config.DEVICE
print(f'Device is: {DEVICE} | REPLICA_MODEL = {config.REPLICA_MODEL} | env = {sys.prefix}')
os.makedirs('results', exist_ok=True)

## Expected values

- **log**: final "Testing with FULL TEST LOADER" line of the run log (= "VAL Stats" of the best-F1 epoch).
- **losses**: "VAL Stats" of that epoch.
- **paper**: Table 8, the row for the selected model.

In [ ]:
SPEC = {
    'fp32': {
        'env': 'pytorch_23', 'epoch': 86, 'n_params': 68882,
        'log': {'Accuracy': [0.9546052813529968, 0.9782483577728271],
                'Precision': [0.9565523862838745, 0.9523926973342896],
                'Recall': [0.9467073678970337, 0.9819888472557068],
                'F1': [0.951604425907135, 0.9669643640518188]},
        'log_losses': {'Total': 11.0675, 'Smoke': 7.07, 'Fire': 3.99},
        'paper': {'smoke': {'Precision': 95.66, 'Recall': 94.67, 'F1': 95.16},
                  'fire': {'Precision': 95.24, 'Recall': 98.20, 'F1': 96.70}, 'F1_macro': 95.93},
        'paper_ref': 'Table 5 (MobileNetV2 Nano, 95.93 %), Table 7 FP32, Table 8 FP32',
    },
    'brevitas': {
        'env': 'pytorch_brevitas', 'epoch': 91, 'n_params': 68914,
        'log': {'Accuracy': [0.9500822424888611, 0.9753289222717285],
                'Precision': [0.9582476615905762, 0.9442546963691711],
                'Recall': [0.9348452091217041, 0.9818620085716248],
                'F1': [0.9464017748832703, 0.9626911878585815]},
        'log_losses': {'Total': 12.5129, 'Smoke': 7.96, 'Fire': 4.56},
        'paper': {'smoke': {'Precision': 95.82, 'Recall': 93.48, 'F1': 94.64},
                  'fire': {'Precision': 94.43, 'Recall': 98.19, 'F1': 96.27}, 'F1_macro': 95.45},
        'paper_ref': 'Table 5 (Quantized MobileNetV2 Nano, 68914 params, 95.45 %), Table 7 Quantized, Table 8 Quantized software',
    },
}[config.REPLICA_MODEL]

env_name = os.path.basename(sys.prefix)
if env_name != SPEC['env']:
    print(f'WARNING: running in {env_name}, original training env was {SPEC["env"]}')

## Configuration check (values from the run log header)

In [ ]:
expected_cfg = dict(NUM_CLASSES=2, CLASSES=['smoke', 'fire'], IMG_H=224, IMG_W=224, BATCH_SIZE=64,
                    LOSS_FN='BCE', SMOKE_PRECISION_WEIGHT=0.8, DS_LEN=None,
                    BREVITAS_MODEL=(config.REPLICA_MODEL == 'brevitas'))
if config.REPLICA_MODEL == 'brevitas':
    expected_cfg.update(WEIGHTS_BIT_WIDTH=4, BIG_LAYERS_WEIGHTS_BIT_WIDTH=4, BIAS_BIT_WIDTH=4, ACTIVATIONS_BIT_WIDTH=4)
for k, v in expected_cfg.items():
    assert getattr(config, k) == v, (k, getattr(config, k), v)
print('config OK')
print('Checkpoint:', config.LOAD_MODEL_FILE, '| original:', config.ORIGINAL_MODEL_FILE)

# Dataloaders

In [ ]:
print("================================= TEST LOADER (paper protocol, drop_last=True) =================================")
test_loader_paper = get_val_loader(return_img_filename=False, drop_last=True)
print("================================= TEST LOADER (full, drop_last=False, with filenames) =================================")
test_loader_full = get_val_loader(return_img_filename=True, drop_last=False)
n_full = len(test_loader_full.dataset)
n_paper = len(test_loader_paper) * config.BATCH_SIZE
print(f'Test images: full = {n_full}, evaluated with drop_last = {n_paper}')
assert n_full == 24371, n_full   # Table 1: 4306 + 4181 + 15884

# Model

In [ ]:
model = MobileNetV2_MINI_RESNET().to(DEVICE)
epoch = load_checkpoint(config.LOAD_MODEL_FILE, model, device=DEVICE)   # load_state_dict(strict=True)
# Brevitas 0.10.2 leaves the quantizer scaling parameters (scaling_impl.value) on CPU after
# load_state_dict; move the whole model to DEVICE again, as the original validate_metrics.ipynb did
# (`.eval(); .to(DEVICE)` before evaluating). No-op for the FP32 model.
model.to(DEVICE)
model.eval()
assert all(t.device.type == torch.device(DEVICE).type for t in list(model.parameters()) + list(model.buffers()))
n_params = sum(p.numel() for p in model.parameters())
ckpt_sha256 = hashlib.sha256(open(config.LOAD_MODEL_FILE, 'rb').read()).hexdigest()
print(f'Class: {type(model).__module__}.{type(model).__name__} | Epoch: {epoch} | Parameters: {n_params} | sha256: {ckpt_sha256}')
assert ckpt_sha256 == config.MODEL_FILE_SHA256, 'local weights differ from the paper checkpoint'
assert epoch == SPEC['epoch'] and n_params == SPEC['n_params']

# A. Training-identical protocol

In [ ]:
loss_fn = BCE_LOSS(device=DEVICE, smoke_precision_weight=config.SMOKE_PRECISION_WEIGHT)
with torch.no_grad():
    losses_A, metrics_A = val_epoch.eval_fn(test_loader_paper, model, loss_fn, DEVICE)
expected_A = SPEC['log']
diff_A = {k: [abs(a - b) for a, b in zip(metrics_A[k], expected_A[k])] for k in expected_A}
max_diff_A = max(max(v) for v in diff_A.values())
print('losses:', losses_A, '| logged:', SPEC['log_losses'])
print('metrics:', metrics_A)
print(f'max |diff| vs log = {max_diff_A:.2e}')

# B. Independent per-image evaluation (full test set)

In [ ]:
rows = []
with torch.no_grad():
    for x, y, files in tqdm(test_loader_full, desc='Predicting'):
        p = torch.sigmoid(model(x.to(DEVICE))).cpu().numpy()
        y = y.numpy()
        for i in range(len(files)):
            rows.append({'file': files[i], 'y_smoke': int(y[i, 0]), 'y_fire': int(y[i, 1]),
                         'p_smoke': float(p[i, 0]), 'p_fire': float(p[i, 1])})
pred = pd.DataFrame(rows)
pred['file'] = pred['file'].str.replace(os.path.expanduser('~/uav/datasets/'), '', regex=False)
assert len(pred) == n_full

In [ ]:
def get_metrics(y_pred, y_true):
    y_pred, y_true = np.asarray(y_pred), np.asarray(y_true)
    TP = int((y_pred * y_true).sum()); FP = int((y_pred * (1 - y_true)).sum())
    TN = int(((1 - y_pred) * (1 - y_true)).sum()); FN = int(((1 - y_pred) * y_true).sum())
    n = TP + FP + TN + FN
    precision = TP / (TP + FP); recall = TP / (TP + FN)
    return {'TP': TP, 'FP': FP, 'TN': TN, 'FN': FN, 'N': n,
            'Accuracy': (TP + TN) / n, 'Precision': precision, 'Recall': recall,
            'F1': 2 * precision * recall / (precision + recall)}

def evaluate(df, threshold=0.5):
    # same binarization as torchmetrics Multilabel* (preds > threshold)
    out = {c: get_metrics((df[f'p_{c}'] > threshold).astype(int), df[f'y_{c}']) for c in ('smoke', 'fire')}
    out['F1_macro'] = (out['smoke']['F1'] + out['fire']['F1']) / 2
    return out

metrics_B_full = evaluate(pred)
metrics_B_paper_subset = evaluate(pred.iloc[:n_paper])
diff_AB = max(abs(metrics_B_paper_subset[c][k] - metrics_A[k][i])
              for i, c in enumerate(('smoke', 'fire')) for k in ('Accuracy', 'Precision', 'Recall', 'F1'))
print(json.dumps({'full_test': metrics_B_full, 'paper_subset': metrics_B_paper_subset}, indent=1))
print(f'max |B(subset) - A| = {diff_AB:.2e}')

# Comparison with the paper

In [ ]:
paper = SPEC['paper']
table = []
for c in ('smoke', 'fire'):
    for k in ('Precision', 'Recall', 'F1'):
        table.append({'class': c, 'metric': k, 'paper_%': paper[c][k],
                      'replica_A_%': round(100 * metrics_A[k][0 if c == 'smoke' else 1], 2),
                      'replica_full_test_%': round(100 * metrics_B_full[c][k], 2)})
f1m_A = 100 * (metrics_A['F1'][0] + metrics_A['F1'][1]) / 2
table.append({'class': 'macro', 'metric': 'F1', 'paper_%': paper['F1_macro'],
              'replica_A_%': round(f1m_A, 2), 'replica_full_test_%': round(100 * metrics_B_full['F1_macro'], 2)})
comparison = pd.DataFrame(table)
comparison['A_matches_paper'] = comparison['paper_%'] == comparison['replica_A_%']
print(SPEC['paper_ref'])
comparison

# Save results and provenance

In [ ]:
def _git(*a):
    return subprocess.run(['git', *a], capture_output=True, text=True).stdout.strip()
pk = ['torch', 'torchvision', 'albumentations', 'torchmetrics', 'numpy', 'opencv-python', 'brevitas']
packages = {}
for p_ in pk:
    try:
        packages[p_] = md.version(p_)
    except md.PackageNotFoundError:
        pass
tag = f'{config.REPLICA_MODEL}__{os.path.basename(config.PAPER_RUN_DIR.rstrip("/"))}__epoch{epoch}'
record = {
    'timestamp': datetime.datetime.now().astimezone().isoformat(timespec='seconds'),
    'replica_model': config.REPLICA_MODEL, 'paper_result': SPEC['paper_ref'],
    'historical_run': config.PAPER_RUN_DIR, 'checkpoint': config.LOAD_MODEL_FILE,
    'checkpoint_original': config.ORIGINAL_MODEL_FILE, 'checkpoint_sha256': ckpt_sha256,
    'checkpoint_epoch': epoch, 'n_params': n_params,
    'model_class': f'{type(model).__module__}.{type(model).__name__}',
    'n_test_full': n_full, 'n_test_paper_protocol': n_paper,
    'A_training_protocol': {'losses': losses_A, 'logged_losses': SPEC['log_losses'], 'metrics': metrics_A,
                            'max_abs_diff_vs_log': max_diff_A},
    'B_full_test': metrics_B_full, 'B_paper_subset': metrics_B_paper_subset, 'max_abs_diff_B_subset_vs_A': diff_AB,
    'comparison_with_paper': comparison.to_dict(orient='records'),
    'env': {'conda_env': env_name, 'python': sys.version.split()[0], 'executable': sys.executable, 'packages': packages,
            'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None},
    'git': {'commit': _git('rev-parse', 'HEAD'), 'dirty': bool(_git('status', '--porcelain'))},
}
with open(f'results/validate_paper_metrics__{tag}.json', 'w') as f:
    json.dump(record, f, indent=2, default=str)
pred.to_csv(f'results/predictions__{tag}__full_test.csv', index=False, float_format='%.6f')
print(f'Saved results/validate_paper_metrics__{tag}.json and predictions CSV')